In [2]:

from pathlib import Path
import pandas as pd
import sqlite3
import math

# Find the main project folder
# BASE_DIR = Path(__file__).resolve().parents[1]

# Load cleaned CSV files
customers = pd.read_csv(BASE_DIR / "data/clean/customers.csv")
products = pd.read_csv(BASE_DIR / "data/clean/products.csv")
orders = pd.read_csv(BASE_DIR / "data/clean/orders.csv")

# -------------------------------
# 1. CUSTOMER VALIDATION
# -------------------------------

assert len(customer) == 1200
assert customer["customer_id"].is_unique
assert customer["city"].nunique() == 6
assert customer["age"].notna().all()
assert customer["age"].between(18, 60).all()
assert set(customer["gender"].unique()) == {"male", "female", "unknown"}

# -------------------------------
# 2. PRODUCT VALIDATION
# -------------------------------

assert len(product) == 40
assert product["product_id"].is_unique
assert product["price_inr"].gt(0).all()

# -------------------------------
# 3. ORDER VALIDATION
# -------------------------------

assert len(order) == 12309
assert order["order_id"].is_unique
assert (order["status"] == "completed").sum() == 11110
assert (order["status"] == "cancelled").sum() == 1199
assert (order["status"] == "pending").sum() == 0
assert order["order_amount"].notna().all()

completed = order[order["status"] == "completed"]
cancelled = order[order["status"] == "cancelled"]

assert completed["delivery_minutes"].notna().all()
assert cancelled["delivery_minutes"].isna().sum() == 1199

total_revenue = completed["order_amount"].sum()
assert math.isclose(total_revenue, 2998668.80, abs_tol=0.01)

# -------------------------------
# 4. SQLITE DATABASE VALIDATION
# -------------------------------

db_path = BASE_DIR / "data/instamart.db"
conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

table_names = set(tables["name"])

assert {"customers", "products", "orders"}.issubset(table_names)

assert pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM customers", conn
).iloc[0]["n"] == 1200

assert pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM products", conn
).iloc[0]["n"] == 40

assert pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM orders", conn
).iloc[0]["n"] == 12309

conn.close()

# -------------------------------
# FINAL RESULT
# -------------------------------

print("Customer checks: PASSED")
print("Product checks: PASSED")
print("Order checks: PASSED")
print("SQLite database checks: PASSED")
print("ALL CHECKPOINTS PASSED")


NameError: name 'BASE_DIR' is not defined